<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Unsupervised Learning - Apriori - Lebensmittel
</b></font> </br></p>

---


[mlxtend](https://github.com/rasbt/mlxtend)

# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten
from pandas import read_csv, DataFrame, concat

# Preprocessing (Transaktionen)
from mlxtend.preprocessing import Transactioncoder

# Modell (Assoziationsanalyse)
from mlxtend.frequent_patterns import apriori, association_rules

# Visualisierung
import plotly.express as px

In [ ]:
# Warnungen ausstellen
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="ipykernel.ipkernel")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client.session")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>
✅ Passenden Transfer-Case als Orientierung auswählen</br>

<p><font color='black' size="5">
Daten lesen
</font></p>


In [ ]:
data = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/basket_01_format_1.csv",
    header=None,)

<p><font color='black' size="5">
EDA (Exploratory Data Analysis)
</font></p>


In [ ]:
data.info()

In [ ]:
data.head(5)

In [ ]:
data.shape

# 2 | Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Duplikate ermitteln/löschen</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>
✅ Preprocessing-Entscheidungen aus dem Transfer-Case auf Übertragbarkeit prüfen</br>

<p><font color='black' size="5">
Datenaufbereitung abhängig vom Format der Quelldaten
</font></p>

+ Format 1: csv-Datei, Item1, Item2, Item3, Item4, ..
+ Format 2: csv-Datei, Transaktions-Id, Item, <weitere Datenfelder wie Datum, Shop,...>
.

In [ ]:
# @title
# @markdown <p><font size="5" color='grey'> <b> Format 1 - ✅ </b></font> </br></p>
data_new = []
for i in range(0, len(data)):  # Für alle Zeilen des DataFrames
    item_list = [str(n) for n in data.iloc[i][:] if str(n) != "nan"]
    data_new.append(list(item_list))

In [ ]:
# @title
# @markdown <p><font size="5" color='grey'> <b> Format 2 - ❌</b></font> </br></p>
# unique_transaction = data.Transaction.unique()
# data_new = []
# for index in unique_transaction:
#   rows = data[data['Transaction'] == index]
#   item_list = [n for n in rows['Item']]
#   data_new.append(list(item_list))

<p><font color='black' size="5">
Erstelle eine True/False DataFrame mit den Items als Spalte
</font></p>

In [ ]:
coder = Transactioncoder()
data_array = coder.fit(data_new).transform(data_new) # Erstellt eine Python-Liste mit True/False Angaben je Item
df = DataFrame(data_array, columns=coder.columns_)
df

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Modellansatz aus dem Transfer-Case als Orientierung für die eigene Lösung nutzen</br>

<p><font color='black' size="5">
Apriori
</font></p>

<p><font size="3">
Der Apriori-Algorithmus ist der beliebteste Algorithmus für das Mining von Assoziationsregeln. Er findet die häufigsten Kombinationen in einem Datenbestand.
<br>



**In der Praxis relevant wenn:**
- Häufig gemeinsam auftretende Kombinationen in Transaktionsdaten gefunden werden sollen (z.B. Warenkorbanalyse, Cross-Selling)
- Die Daten als Transaktionslisten vorliegen (Ereignisse ohne feste Spaltenstruktur)
- Assoziationsregeln mit Konfidenz und Lift bewertet werden sollen

**Nicht geeignet wenn:**
- Die Datenmenge sehr groß ist → Apriori ist bei niedrigem `min_support` ineffizient; dann FP-Growth verwenden
- Vorhersagen für einzelne Kunden gefragt sind → Apriori findet globale Muster, keine individualisierten

**Typischer Fehler:**
`min_support` zu niedrig setzen. Dann explodiert die Anzahl der Itemsets und die Laufzeit. Immer zuerst mit hohem Support beginnen und schrittweise senken.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Prozess</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    TRANS[Transaktionsdaten] --> ENC["Transactioncoder<br/>True/False Matrix"]
    ENC --> FI["Apriori<br/>Häufige Itemsets"]
    FI --> AR[Assoziationsregeln]
    AR --> FILTER["Filtern nach<br/>Support / Confidence / Lift"]
    FILTER --> OUT[Regelwerk]

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;

    %% Zuweisung
    class TRANS input;
    class OUT output;
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=950))

# 4 | Evaluate
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Cross-Validation</br>
✅ Bootstrapping</br>
✅ Regularisierung prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>
✅ Ergebnisse mit dem Transfer-Case vergleichen und Unterschiede erklären</br>

<p><font color='black' size="5">
Parameter-Set
</font></p>

In [ ]:
# Übergreifende Parameter
min_support = 0.01
min_confidence = 0.5

# Spaltenauswahl
auszug = [
    "antecedents",
    "consequents",
    "antecedent support",
    "consequent support",
    "support",
    "confidence",
    "lift",
]
auszug_num = [
    "antecedent support",
    "consequent support",
    "support",
    "confidence",
    "lift",
]
auszug_confidence = [
    "antecedents",
    "consequents",
    "antecedent support",
    "consequent support",
    "support",
    "confidence",
]
auszug_lift = ["antecedents", "consequents", "lift"]

<p><font color='black' size="5">
Berechnung
</font></p>

In [ ]:
# Berechnung Frequenzen
frequent_itemsets = apriori(df, min_support=min_support, use_colnames=True)
frequent_itemsets = frequent_itemsets[["itemsets", "support"]]

In [ ]:
# Berechnung Regeln
associationRulesSet = association_rules(
    frequent_itemsets, num_itemsets=len(df), metric="confidence", min_threshold=min_confidence
)
associationRulesSet = associationRulesSet[auszug]

# Support und Confidence in % umrechnen; Lift bleibt als Verhältnis
for col in ["antecedent support", "consequent support", "support", "confidence"]:
    associationRulesSet[col] = (associationRulesSet[col] * 100).round(2)
associationRulesSet["lift"] = associationRulesSet["lift"].round(2)

**Support**


Der Support sagt uns, wie häufig ein Item oder eine Kombination von Items  gekauft wird. Er wird als Prozentsatz aller Transaktionen angegeben.

```WENN man alle Transaktionen betrachtet, DANN ist in X% der Transaktionen Produkt A enthalten```


Beispiel:    
Wenn in 100 Einkäufen 20-mal Brot und Butter zusammen gekauft wurden, dann ist der Support für diese Kombination 20%.

In [ ]:
frequent_itemsets.head(5)

In [ ]:
frequent_itemsets.min()

In [ ]:
frequent_itemsets.max()

**Confidence**

Die Confidence zeigt, wie verlässlich eine Regel ist. Sie gibt an: Wenn Produkt A (*antecedents*) gekauft wird, wie wahrscheinlich ist es dann, dass auch Produkt B (*consequents*) gekauft wird?

```WENN Produkt A in einer Transaktion enthalten ist, DANN ist mit X% Wahrscheinlichkeit auch Produkt B in dieser Transaktion enthalten```

Beispiel:    
Wenn von 50 Kunden, die Brot kaufen, 40 auch Butter kaufen, dann ist die Confidence 80%. Die Regel "Wenn Brot, dann Butter" hat also eine hohe Verlässlichkeit.

In [ ]:
associationRulesSet[auszug_confidence].head(5)

In [ ]:
associationRulesSet[auszug_confidence].tail(5)

**Lift**

Der Lift zeigt, ob zwei Produkte ***wirklich*** zusammenhängen oder ob sie nur ***zufällig*** zusammen gekauft werden.

+ Lift = 1 → **keine** Beziehung, Produkte werden unabhängig voneinander gekauft
+ Lift > 1 → eine **positive** Beziehung/Regel, Produkte werden häufiger zusammen gekauft
+ Lift < 1 → **negative** Beziehung zwischen den Produkten, gegenseitig ausschließende oder substituierende Produkte


```WENN der Wert größer als 1 ist, DANN werden die Produkte A und B häufiger zusammen gekauft```

Beispiel:     
Ein Lift von 2,5 zwischen Brot und Butter bedeutet, dass diese Kombination 2,5-mal häufiger vorkommt als bei zufälligen Einkäufen zu erwarten wäre. Die Produkte haben also eine echte Verbindung zueinander.

In [ ]:
associationRulesSet[auszug_lift].head(5)

In [ ]:
associationRulesSet[auszug_lift].tail(5)

Confidence und Lift messen die **Stärke** einer Assoziationsregel. Die **Confidence** gibt an, wie oft die Regel stimmt, wenn die Bedingung erfüllt ist. **Lift** vergleicht diese Wahrscheinlichkeit mit der Wahrscheinlichkeit, dass die Konsequenz zufällig eintritt. Ein Lift von über 1 bedeutet, dass die Regel aussagekräftiger ist als ein zufälliges Auftreten.

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>
✅ Grenzen der Übertragbarkeit vom Transfer-Case auf den eigenen Anwendungsfall dokumentieren</br>